## Lunar lander with DQN-style neural function approximator using PyTorch
### Christian Igel, 2026

If you have suggestions for improvement, [let me know](mailto:igel@diku.dk).

Imports:

In [1]:
import gymnasium as gym

from tqdm.notebook import tqdm  # Progress bar

import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt

import time

Define *Q* network architecture:

In [2]:
class QNetwork(nn.Module):
    def __init__(self, state_size=8, action_size=4, hidden_size=10, bias=True):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size, bias)  
        self.fc2 = nn.Linear(hidden_size, hidden_size, bias)  
        self.output_layer = nn.Linear(hidden_size + state_size, action_size, bias)

    def forward(self, x_input):
        x = F.tanh(self.fc1(x_input))
        x = F.tanh(self.fc2(x))
        x = torch.cat((x_input, x), dim=1)
        x = self.output_layer(x)
        return x

Data structure for storing experiences:

In [3]:
class Memory():
    def __init__(self, state_dim, max_size = 1000):
        self.states      = np.zeros([max_size, state_dim], dtype=np.float32)
        self.next_states = np.zeros([max_size, state_dim], dtype=np.float32)
        self.actions     = np.zeros(max_size, dtype=np.int64)
        self.rewards     = np.zeros(max_size, dtype=np.float32)
        self.dones       = np.zeros(max_size, dtype=np.int64)
        self.max_size = max_size
        self.ptr      = 0
        self.size     = 0
    
    def add(self, state, action, reward, next_state, done):
        self.states[self.ptr]      = state
        self.next_states[self.ptr] = next_state
        self.actions[self.ptr]     = action
        self.rewards[self.ptr]     = reward
        self.dones[self.ptr]       = done
        self.ptr = (self.ptr + 1) % self.max_size
        self.size = min(self.size + 1, self.max_size)
        
    def sample_tensors(self, batch_size):
        idxs = np.random.choice(self.size, 
                               size=batch_size, 
                               replace=False)
        return torch.as_tensor(self.states[idxs]), torch.as_tensor(self.actions[idxs]), torch.as_tensor(self.rewards[idxs]), torch.as_tensor(self.next_states[idxs]), torch.as_tensor(self.dones[idxs])
        

Q-learning:

In [4]:
def doit(trial, 
         env,
         state_size,
         action_size,
         double=True,
         train_episodes = 250,           # Max number of episodes to learn from
         gamma = 0.99,                   # Future reward discount
         learning_rate = 0.001,          # Q-network learning rate
         tau = .01,                      # learning rate for target network
         explore_start = 1.0,            # Exploration probability at start
         explore_stop = 0.0001,          # Minimum exploration probability 
         decay_rate = 0.05,              # Exponential decay rate for exploration prob
         hidden_size = 64,               # Number of units in each Q-network hidden layer
         memory_size = 10000,            # Memory capacity
         batch_size = 128,               # Experience mini-batch size
         seed = 41
        ):
    # Return values
    total_rewards = np.zeros(train_episodes)
    
    # Initialize the simulation
    state = env.reset(seed=seed+trial)[0]

    # Experience replay buffer
    memory = Memory(state_dim=state_size, max_size=memory_size)
    
    # Make a bunch of random actions and store the experiences
    pretrain_length = batch_size   # Number experiences to pretrain the memory
    for _ in tqdm(range(pretrain_length)):
        # Make a random action
        action = env.action_space.sample()
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Add experience to memory
        memory.add(state, action, reward, next_state, done)

        if done:
            # Start new episode
            env.reset()
            # Take one random step to get the pole and cart moving
            state, reward, terminated, truncated, _ = env.step(env.action_space.sample())
        else:
            state = next_state
    
    # Initialize Q networks
    Q_online = QNetwork(hidden_size=hidden_size)
    Q_target = QNetwork(hidden_size=hidden_size)
    Q_target.load_state_dict(Q_online.state_dict())
    
    # Optimizer and loss
    optimizer = torch.optim.AdamW(Q_online.parameters(), lr=learning_rate) # AdamW uses weight decay by default
    loss_fn = torch.nn.MSELoss(reduction='none')
    
    # Loop over episodes
    for ep in range(train_episodes):
        total_reward = 0  # Return / accumulated rewards
        state = env.reset()[0]  # Reset and get initial state
        while True:
            # Explore or exploit
            explore_p = explore_stop + (explore_start - explore_stop) * np.exp(-decay_rate*ep)
            if explore_p > np.random.rand():
                # Pick a random action
                action = env.action_space.sample()
            else:
                # Get action from Q-network
                with torch.no_grad():
                    state_tensor = torch.as_tensor(state).view(1, -1)
                    Qs = Q_online(state_tensor)
                    action = torch.argmax(Qs).item()
    
            # Take action, get new state and reward
            next_state, reward, terminated, truncated, _ = env.step(action)
            # Both termination and truncation indicate failure 
            done = terminated or truncated

            # Add experience to memory
            memory.add(state, action, reward, next_state, done)
            total_reward += reward  # Return / accumulated rewards
               
            if done:
                print('Episode: {}'.format(ep), 'Total reward: {}'.format(total_reward),
                      'Training loss: {:.4f}'.format(loss), 'Explore P: {:.4f}'.format(explore_p))
                total_rewards[ep] = total_reward
                break; # End of episode
            else:
                state = next_state
                
            # Sample mini-batch from memory
            states, actions, rewards, next_states, dones  = memory.sample_tensors(batch_size)
                  
            # Compute Q values for all actions in the new state       
            target_Qs = Q_target(next_states)
    
            # Transitions to failure state get zero reward
            mask = 1 - dones
    
            # Do (double) Q-learning
            with torch.no_grad():
                q_target_next = Q_target(next_states)
                if(double):
                    q_online_next = Q_online(next_states)
                    argmax_indices = torch.argmax(q_online_next, axis=1, keepdim=True)
                else:
                    argmax_indices = torch.argmax(q_target_next, axis=1, keepdim=True)
                target = q_target_next.gather(1, argmax_indices).squeeze(1).detach() * mask
                # Compute targets
                y = rewards + gamma * target
            
            # Network learning starts here
            optimizer.zero_grad()
            
            # Compute the Q values of the actions taken        
            q_online = Q_online(states)  # Q values for all action in each state
            Q = q_online.gather(1, actions.unsqueeze(-1)).squeeze()  # Only the Q values for the actions taken
            
            # Gradient-based update
            elementwise_loss = loss_fn(Q, y)
            loss = torch.mean(elementwise_loss)
            loss.backward()
            optimizer.step()

            # Update target network
            with torch.no_grad():
                for key, param in Q_online.state_dict().items():
                    Q_target.state_dict()[key].mul_(1 - tau).add_(tau * param)
    return total_rewards

Run experiments:

In [5]:
env = gym.make('LunarLander-v3')
action_size = 4
state_size = 8

no_trials = 10
train_episodes = 250

taus = (0.5,)

start = time.time()
total_rewards_DQN = np.zeros((no_trials, train_episodes))
total_rewards_DDQN = np.zeros((no_trials, train_episodes))
for tau in taus:
    for t in range(no_trials):
        total_rewards_DQN[t] = doit(t, env, state_size, action_size, False, train_episodes = train_episodes, tau=tau, memory_size=8192)
        total_rewards_DDQN[t] = doit(t, env, state_size, action_size, True, train_episodes = train_episodes, tau=tau, memory_size=8192)
    
np.save("DQN_lecture_%d_tau%g.npy" % (no_trials, tau), total_rewards_DQN)
np.save("DDQN_lecture_%d_tau%g.npy" % (no_trials, tau), total_rewards_DDQN)

end = time.time()
print("time: ", end - start)

  0%|          | 0/128 [00:00<?, ?it/s]

Episode: 0 Total reward: -156.01781562063263 Training loss: 82.2355 Explore P: 1.0000
Episode: 1 Total reward: -161.65902850500754 Training loss: 77.8928 Explore P: 0.9512
Episode: 2 Total reward: -98.90131156395562 Training loss: 88.7304 Explore P: 0.9048
Episode: 3 Total reward: -54.57620954566346 Training loss: 6.2821 Explore P: 0.8607
Episode: 4 Total reward: -83.8700950595896 Training loss: 146.0855 Explore P: 0.8187
Episode: 5 Total reward: -94.73164005323319 Training loss: 9.8972 Explore P: 0.7788
Episode: 6 Total reward: -62.73140493123731 Training loss: 37.2150 Explore P: 0.7408
Episode: 7 Total reward: -91.36671703908716 Training loss: 6.8462 Explore P: 0.7047
Episode: 8 Total reward: -211.1432331303098 Training loss: 28.7607 Explore P: 0.6704
Episode: 9 Total reward: -590.18335888205 Training loss: 5.3425 Explore P: 0.6377
Episode: 10 Total reward: -89.1088734239046 Training loss: 27.1408 Explore P: 0.6066
Episode: 11 Total reward: -258.6020281429536 Training loss: 49.7692 E

  0%|          | 0/128 [00:00<?, ?it/s]

Episode: 0 Total reward: -63.69374656169109 Training loss: 78.8852 Explore P: 1.0000
Episode: 1 Total reward: -107.36936218750333 Training loss: 80.0129 Explore P: 0.9512
Episode: 2 Total reward: -50.80728726833992 Training loss: 71.0246 Explore P: 0.9048
Episode: 3 Total reward: -10.94060609342283 Training loss: 312.7872 Explore P: 0.8607
Episode: 4 Total reward: -356.9779782469541 Training loss: 84.1853 Explore P: 0.8187
Episode: 5 Total reward: 13.407083347257057 Training loss: 157.0037 Explore P: 0.7788
Episode: 6 Total reward: -125.74064397815992 Training loss: 5.6426 Explore P: 0.7408
Episode: 7 Total reward: -244.26767939685217 Training loss: 68.0887 Explore P: 0.7047
Episode: 8 Total reward: -111.39556933065478 Training loss: 7.3748 Explore P: 0.6704
Episode: 9 Total reward: -338.43551228207536 Training loss: 6.3523 Explore P: 0.6377
Episode: 10 Total reward: -219.48622324112824 Training loss: 274.8050 Explore P: 0.6066
Episode: 11 Total reward: -191.15807768495358 Training los

  0%|          | 0/128 [00:00<?, ?it/s]

...

Episode: 0 Total reward: -439.3781692525423 Training loss: 14.5526 Explore P: 1.0000
Episode: 1 Total reward: -93.2462766728312 Training loss: 12.6733 Explore P: 0.9512
Episode: 2 Total reward: -82.62929910649832 Training loss: 144.5346 Explore P: 0.9048
Episode: 3 Total reward: -327.3921344118617 Training loss: 11.3219 Explore P: 0.8607
Episode: 4 Total reward: -328.35986521386167 Training loss: 12.5473 Explore P: 0.8187
Episode: 5 Total reward: -56.96984718367213 Training loss: 154.2429 Explore P: 0.7788
Episode: 6 Total reward: -338.17247557347065 Training loss: 20.7397 Explore P: 0.7408
Episode: 7 Total reward: -242.8477961375376 Training loss: 132.3065 Explore P: 0.7047
Episode: 8 Total reward: -332.05693728356766 Training loss: 8.5604 Explore P: 0.6704
Episode: 9 Total reward: -107.68782838732213 Training loss: 60.1823 Explore P: 0.6377
Episode: 10 Total reward: -89.4909468884005 Training loss: 194.9463 Explore P: 0.6066
Episode: 11 Total reward: -211.4180283707384 Training loss:

  0%|          | 0/128 [00:00<?, ?it/s]

Episode: 0 Total reward: -274.1634604515715 Training loss: 117.1714 Explore P: 1.0000
Episode: 1 Total reward: -174.57729211785545 Training loss: 66.6026 Explore P: 0.9512
Episode: 2 Total reward: -210.35940919542426 Training loss: 126.9367 Explore P: 0.9048
Episode: 3 Total reward: -73.91587774285793 Training loss: 58.3085 Explore P: 0.8607
Episode: 4 Total reward: -145.37929561713267 Training loss: 38.8290 Explore P: 0.8187
Episode: 5 Total reward: -295.54980761106447 Training loss: 134.2699 Explore P: 0.7788
Episode: 6 Total reward: -319.3438093445629 Training loss: 7.1683 Explore P: 0.7408
Episode: 7 Total reward: -176.6028509140266 Training loss: 46.4266 Explore P: 0.7047
Episode: 8 Total reward: -253.70360691746458 Training loss: 78.2804 Explore P: 0.6704
Episode: 9 Total reward: -161.32154866667986 Training loss: 90.2864 Explore P: 0.6377
Episode: 10 Total reward: -126.48161161801225 Training loss: 48.0564 Explore P: 0.6066
Episode: 11 Total reward: -227.29554243187994 Training 

  0%|          | 0/128 [00:00<?, ?it/s]

Episode: 0 Total reward: -174.38112391698064 Training loss: 71.0552 Explore P: 1.0000
Episode: 1 Total reward: -281.2590905407701 Training loss: 59.6234 Explore P: 0.9512
Episode: 2 Total reward: -181.65847213779838 Training loss: 59.3929 Explore P: 0.9048
Episode: 3 Total reward: -145.46754779146033 Training loss: 149.6422 Explore P: 0.8607
Episode: 4 Total reward: -457.8202231701332 Training loss: 51.0731 Explore P: 0.8187
Episode: 5 Total reward: -94.1850312469248 Training loss: 88.4961 Explore P: 0.7788
Episode: 6 Total reward: -159.08616175825495 Training loss: 8.0610 Explore P: 0.7408
Episode: 7 Total reward: -158.30893838325898 Training loss: 3.7672 Explore P: 0.7047
Episode: 8 Total reward: -177.109189076026 Training loss: 112.3670 Explore P: 0.6704
Episode: 9 Total reward: -209.3744190706429 Training loss: 3.2924 Explore P: 0.6377
Episode: 10 Total reward: -152.38358563949816 Training loss: 40.1246 Explore P: 0.6066
Episode: 11 Total reward: -256.2094909371436 Training loss: 1

Make a plot

In [6]:
# Moving average for smoothing plot
def running_mean(x, N):
    cumsum = np.cumsum(np.insert(x, 0, x[0]*np.ones(N)))
    return (cumsum[N:] - cumsum[:-N]) / N

# Load data and compute percentiles
def compute_stats(filename):
    try:
        data = np.load(filename)
    except:
        print("exception when trying to load ", filename)
    p10 = np.percentile(data, 10, axis=0)
    p50 = np.percentile(data, 50, axis=0)
    p90 = np.percentile(data, 90, axis=0)
    return p10, p50, p90

p10, p50, p90                      = compute_stats("DQN_lecture_%d_tau%g.npy" % (no_trials, tau))
p10_double, p50_double, p90_double = compute_stats("DDQN_lecture_%d_tau%g.npy" % (no_trials, tau))


In [1]:
# Smoothing
window = 5
# Create a wide figure and share the y-axis between the two subplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True);

axes[0].grid(True);
x = np.arange(p50.size) + 1
line, = axes[0].plot(x, running_mean(p50, window), label=rf"DQN, $\tau=$ {tau}")
axes[0].fill_between(x, running_mean(p10, window), running_mean(p90, window), color=line.get_color(), alpha=0.2)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Accumulated Reward')
axes[0].legend()

axes[1].grid(True);
x = np.arange(p50_double.size) + 1
line, = axes[1].plot(x, running_mean(p50_double, window), label=rf"DDQN, $\tau=$ {tau}")
axes[1].fill_between(x, running_mean(p10_double, window), running_mean(p90_double, window), color=line.get_color(), alpha=0.2)
axes[1].set_xlabel('Episode')
axes[1].legend()

y_min = min(np.min(m) for m in list(running_mean(p10_double, window)) + list(running_mean(p10, window)))
y_max = max(np.max(m) for m in list(p90) + list(p90_double))
# Add a small margin
pad = 0.02 * (y_max - y_min if y_max > y_min else 1.0)
axes[0].set_ylim(y_min - pad, y_max + pad)


plt.tight_layout()
#plt.show();
plt.savefig('deepQ.pdf');

NameError: name 'plt' is not defined